# An introduction to explainability in ML

In [ ]:
! pip install torch --quiet

In [ ]:
import sklearn
import pandas as pd
from pathlib import Path
import requests
from folktables import ACSDataSource, ACSEmployment
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import make_pipeline
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
from torch.nn import Module, Sequential, Linear, ReLU
import matplotlib.pyplot as plt


# Exercise 1: Global explanations [25 mins]

We saw in the lecture that one way to explain a complex model is by approximating it with an inherently explainable model like decision trees.

In this exercise, you will build on the exercise from the last lecutre. Write a function that takes one of our three datasets (the custom synthetic dataset, COMPAS or ACS PUMS) and trains one of our two models (Logistic Regression and a Neural Network). The output of this function is the trained model.

Now write another function that takes a trained model and explains it using a decision tree. Which model is easier to explain with the decision tree?

In [ ]:
def load_custom_data():
    x, y = sklearn.datasets.make_classification(n_samples=200, n_features=10, n_informative=2, n_redundant=8, n_repeated=0, random_state=710)
    return x, y, [f"f{i}" for i in range(x.shape[1])]

x, y, names = load_custom_data()
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=11)

In [ ]:
def load_compas():

    # Based on: https://fairlens.readthedocs.io/en/latest/user_guide/compas.html
    url = "https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv"
    local_name = Path("compas-scores-two-years.csv")
    if not local_name.is_file():
        response = requests.get(url)
        with open("compas-scores-two-years.csv", "w") as f:
            f.write(response.content.decode("utf-8"))
    df = pd.read_csv(local_name)
    df = df.sample(frac=1, random_state=1)

    df = df[(df["days_b_screening_arrest"] <= 30)
            & (df["days_b_screening_arrest"] >= -30)
            & (df["is_recid"] != -1)
            & (df["c_charge_degree"] != 'O')
            & (df["score_text"] != 'N/A')].reset_index(drop=True)
    selected_cols = [  # The features that we will use for classification
    "age",
    "juv_fel_count",
    "juv_misd_count",
    "juv_other_count",
    "priors_count",
    "c_charge_degree",
    ]
    sens = "race"  #  Will not use for classification but use it for fairness analysis
    label = "two_year_recid"  # The label we will try to predict

    df_selected = df[selected_cols + [sens] + [label]]
    y = df_selected[label].to_numpy()
    df_selected.drop(columns=[label, sens], inplace=True)
    df_one_hot = pd.get_dummies(df_selected)
    x = df_one_hot.to_numpy()
    return x, y, df_one_hot.columns

def load_pums():
    data_source = ACSDataSource(survey_year='2018', horizon='1-Year', survey='person')
    acs_data = data_source.get_data(states=["AL"], download=True)  # Limiting to AL. You can try another state or all the states.
    x, y, _ = ACSEmployment.df_to_numpy(acs_data)  # The group in this case is the race. It is also included in the features.
    return x, y, ACSEmployment.features


In [ ]:
from sklearn.neural_network import MLPClassifier

# Your code here
def train_model(model, x_train, y_train, x_test, y_test, scaler=None):
    if scaler is not None:
        model = make_pipeline(scaler, model)
    model.fit(x_train, y_train)
    print(f"Test accuracy: {model.score(x_test, y_test):0.2f}")
    return model

# Your code here
x, y, names = load_pums()
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=1)
#model = LogisticRegression(max_iter=1000)
model = MLPClassifier(hidden_layer_sizes=[64,64])
model = train_model(model, x_train, y_train, x_test, y_test, scaler=MinMaxScaler())

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

pred_train = model.predict(x_train)
pred_test = model.predict(x_test)

explainer_model = DecisionTreeClassifier(max_depth=10, random_state=21)
explainer_model = train_model(explainer_model, x_train, pred_train, x_test, pred_test)

def sort_and_print_features(names, scores):
    names_scores = sorted(zip(names, scores), key=lambda x: x[1], reverse=True)
    for n, s in names_scores:
        print(f"{n: <10} {s:0.3f}")

sort_and_print_features(names, explainer_model.feature_importances_)

In [ ]:
plot_tree(explainer_model, max_depth=2, fontsize=5, feature_names=names, proportion=True)

# Exercise 2: Evaluating explanations [25 mins]

In the lecture we saw that we can compute the quality of local explanations by dropping most or least important features and observing the impact on the model outputs. 

Given a dataset, e.g., PUMS and a model, e.g., a Neural Net, compare the following two explainability methods:
1. SHAP
2. A dummy explainability method that assigns an importance score to each feature by sampling from `np.random.rand()`

When dropping a feature, you can measure the difference in model output as $|f(\mathbf{x}) - f(\bar{\mathbf{x}})|$ where $\mathbf{x}$ is the original input and $\bar{\mathbf{x}}$ is the input with the feature removed.

How do the two explainers compare based on your performance metric?


In [ ]:
# Your code here
import shap

def generate_explanations(explainer, sample_to_explain):
    svs = explainer.shap_values(sample_to_explain, nsamples=100)
    return svs

def drop_feature_and_compute_change(x, importance_scores, baseline, fx, num_drop, bottom=False):
    x = x.reshape(1, x.shape[-1])  # 1 row, num_features columns
    baseline = baseline.reshape(1, baseline.shape[-1])
    idxs = np.argsort(np.abs(importance_scores.flatten()))
    if not bottom:
        idxs = idxs[::-1] # Reverse the array
    idxs = idxs[:num_drop]

    x_perturbed = x.copy().flatten()
    x_perturbed[idxs] = baseline.flatten()[idxs]
    x_perturbed = x_perturbed.reshape(1,-1)
    pred_original = fx(x)[:,1]
    pred_perturbed = fx(x_perturbed)[:,1]

    return np.abs(pred_original - pred_perturbed)

fx = model.predict_proba
baseline = x_train.mean(axis=0)  # Replace the delated feature with these values
explainer = shap.KernelExplainer(fx, baseline.reshape(1,-1))

np.random.seed(11234)

sample_to_explain = x_test[0]
svs = generate_explanations(explainer, sample_to_explain)
svs = svs[:,1]
num_drop = 3

change = drop_feature_and_compute_change(
    sample_to_explain, 
    #svs.reshape(-1,1),
    np.random.rand(x.shape[1]),
    baseline, 
    fx, 
    num_drop, 
    bottom=False)

print(change)

# Exercise 3: Training Neural Networks with Gradient Descent [20 mins]

In this exercise, you do not have to solve much. Your task is to familiarize yourself with the code for training a neural network in PyTorch.

Walk through the code and make sure you understand it well.

Then train the network on PUMS data.

Neural networks are trained using graient descent. We pass the data in batches. The code below sets up a Data Loader that can let you iterate over the data in batches.

In [ ]:
x, y, names = load_pums()
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=1)

batch_size = 16

def convert_x_y_to_tensors(x, y):
    # Takes numpy arrays and convert them to PyTorch tensors.
    # Torch expects data to be in torch.Tensor objects
    x = torch.from_numpy(x)
    x = x.to(dtype=torch.float) # Features are floats
    y = torch.from_numpy(y).long()
    return x, y

# TODO: Add code to split the data in train and test sets
x_train, y_train = convert_x_y_to_tensors(x_train, y_train)
x_test, y_test = convert_x_y_to_tensors(x_test, y_test)


# We only make the data loader for the train set
# Test and validation set (if available) are small so we can run the model in a full batch setting
# You can similarly convert them to data loaders if the datasets are too large
train_loader = DataLoader(
    TensorDataset(x_train, y_train), 
    batch_size=batch_size, 
    shuffle=True,             # Shuffle at every epoch
)

for batch in train_loader:
    x_batch, y_batch = batch  # Unpack the batch
    print(f"Batch shape is x: {x_batch.shape} -- y: {y_batch.shape}")

Set up the model

In [ ]:
# All neural networks should inherit from nn.Module which implements some basic methods
class MyFirstNN(Module):
    def __init__(self, n_features, n_classes, hidden_units):     # Operations to carry out when we initialize the model
        super(MyFirstNN, self).__init__()                        # Call the __init__ of the super, or parent, class
        # Let us construct the layers one by one
        layers = []
        n_in = n_features
        for n_out in hidden_units:
            layers.append(Linear(n_in, n_out))  # Recall that a layer is just a mapping from an input to the output
            layers.append(ReLU())
            n_in = n_out
        layers.append(Linear(n_in, n_classes))

        # Next, tell pytorch that the layers are applied in sequence
        self.layers = Sequential(*layers)  # The * operator just unpacks all the layers
        
    def forward(self, x):
        # You should define the behavior of your NN when applied to the data.
        # In this case, we simply apply the layers one after the other.
        # The outputs are the logits, that is, the real-values scores for each class.
        return self.layers(x)
    
    def predict(self, x):
        # This function gives the prediction label.
        with torch.no_grad():
            if isinstance(x, np.ndarray):  # PyTorch models always expect tensors.
                x = torch.FloatTensor(x)
            preds = self(x)  # Apply the model
            pred_class = preds.argmax(axis=1)  # We get the logits for each class, select the one with the highest score
        return pred_class

my_first_nn = MyFirstNN(n_features=x_train.shape[1], n_classes=2, hidden_units=[1024, 1024])
print(my_first_nn)

Check the accuracy. Of course the model is untrained so will have low accuracy.

In [ ]:
def compute_accuracy(model, x, y):
    preds = model.predict(x)
    return (preds==y).sum() / preds.shape[0]

print(f"Train accuracy: {compute_accuracy(my_first_nn, x_train, y_train):0.2f}")

Set up an optimizer 

In [ ]:
learning_rate = 1e-2  # Try different values
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(
    my_first_nn.parameters(),  # The parameters that we should optimize
    lr=learning_rate
)

Train the model

In [ ]:
%matplotlib inline

from IPython import display

num_epochs = 10

def train_model(num_epochs):
    train_accs = []
    my_first_nn.train()  # Put the model in the trianing mode
    
    plt.figure()
    ax = plt.gca()
    
    for epoch in range(num_epochs):
        # -----------------------------------------------
        # Print the performance at the end of each epoch
        # -----------------------------------------------
        my_first_nn.eval()  # Put the model in evaluation mode
        train_accs.append(compute_accuracy(my_first_nn, x_train, y_train))  # If the trainig data is large, you may want to do this iteratively with a data loader
        ax.clear()
        ax.plot(range(epoch+1), train_accs, label="Train accuracy")
        plt.legend()
        display.display(plt.gcf())
        display.clear_output(wait=True)
        my_first_nn.train() # Put the model back in training model
        
        # --------------------------------------------
        # Train the model on all batches in the epoch
        # --------------------------------------------
        for batch in train_loader:
            x_batch, y_batch = batch  # Unpack the batch
    
            # Compute the loss
            pred = my_first_nn(x_batch)
            loss = loss_fn(pred, y_batch)
    
            # Take a gradient step
            loss.backward()        # Compute the gradient
            optimizer.step()       # Update the parameters
            optimizer.zero_grad()  # Zero out the gradients for the next iteration

train_model(num_epochs)

# Exercise 4: Counterfactual explanations [45 mins]

Write a function that given an input and the trained neural network model, generates a counterfactual explanation.

Recall that the counterfactual explanation ($\mathbf{x}_c$) is a modified version of the original input ($\mathbf{x}$) such that $f(\mathbf{x}) \neq f(\mathbf{x}_c)$.

You can generate the counterfactual explanations via gradient descent.

In [ ]:
x_selected = x_test[0]
y_pred = my_first_nn.predict(x_selected.reshape(1,-1))
y_target = torch.abs(1-y_pred)

x_c = x_selected.clone().reshape(1,-1)
x_c.requires_grad_(True)

print("Model pred:", y_pred.item())
print("Counterfactual target:", y_target.item())


learning_rate = 1e-1  # Try different values
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(
    [x_c],  # The parameters that we should optimize
    lr=learning_rate
)

max_iter = 1000
for iter_num in range(max_iter):

    # Forward pass
    pred = my_first_nn(x_c)
    if pred.flatten().argmax(-1).item() == y_target.item():
        print(f"Found counterfactual input in iteration # {iter_num}")
        break

    # Compute loss
    loss = loss_fn(pred, y_target)

    # Take a gradient step
    loss.backward()        # Compute the gradient
    optimizer.step()       # Update the parameters
    optimizer.zero_grad()  # Zero out the gradients for the next iteration




In [ ]:
original = x_selected.flatten().detach().numpy()
counterfactual = x_c.flatten().detach().numpy()
for fo, fc, n in zip(original, counterfactual, names):
    print(f"{n: <10} | {fo:0.2f} | {fc:0.2f}")

